# Auralis Quickstart (Google Colab)

This notebook provisions Auralis on top of **vLLM 0.14.x (V1 Engine)** with PyTorch 2.8 and CUDA 12.9 wheels.

**Requirements:** a Colab runtime with a CUDA GPU (T4 / L4 / A100). On CPU-only runtimes this notebook will error out at import time.

## 1. Install dependencies

The block below:

1. Removes any pre-installed `torch` / `torchvision` / `torchaudio` / `vllm` (Colab frequently ships incompatible builds).
2. Installs PyTorch `2.8.0` (+ `torchvision 0.23.0`, `torchaudio 2.8.0`) with CUDA 12.9 wheels — matching vLLM 0.14's build matrix.
3. Installs vLLM `>=0.14.0` (V1 Engine enabled by default in this release).
4. Installs Auralis and its runtime dependencies.
5. Prints a summary so you can verify the environment.

In [ ]:
# === Auralis + vLLM V1 Engine (Colab setup) ===
# vLLM 0.14 (V1 Engine), PyTorch 2.8, CUDA 12.9 wheels.

# 1. Remove any torch/vllm versions Colab may have preinstalled.
!pip uninstall -y torch torchvision torchaudio vllm || true

# 2. PyTorch 2.8.0 + matching torchvision/torchaudio for CUDA 12.9.
!pip install --index-url https://download.pytorch.org/whl/cu129 \
    torch==2.8.0 \
    torchvision==0.23.0 \
    torchaudio==2.8.0

# 3. vLLM 0.14.x (V1 Engine is the default in this release).
!pip install "vllm>=0.14.0" --extra-index-url https://download.pytorch.org/whl/cu129

# 4. Auralis + TTS dependencies (skip re-resolving torch via --no-deps).
!pip install "safetensors>=0.4.5"
!pip install --no-deps "auralis @ git+https://github.com/JasperSeling/Auralis.git"
!pip install aiofiles langid beautifulsoup4 cachetools colorama cutlet EbookLib einops \
    ffmpeg fsspec hangul_romanize huggingface_hub ipython librosa networkx num2words \
    opencc packaging pyloudnorm pypinyin sounddevice soundfile "spacy==3.7.5" setuptools \
    tokenizers transformers nvidia-ml-py numpy

# 5. Sanity check.
import torch, vllm
print("torch:", torch.__version__, "CUDA:", torch.version.cuda,
      "device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("vllm:", vllm.__version__)

# 6. Declaratively assert V1 Engine (no-op in 0.14 since V1 is the only backend).
import os
os.environ.setdefault("VLLM_USE_V1", "1")

> **Troubleshooting.** If the Colab runtime upgrades its CUDA driver to 13.x, replace both `cu129` substrings above with `cu130` — the other pins remain valid. If pip complains about incompatible resolver constraints after step 4, rerun step 1 first.

## 2. Load the XTTSv2 model

In [ ]:
from auralis import TTS, TTSRequest

tts = TTS().from_pretrained("AstraMindAI/xttsv2")

## 3. Generate speech

Upload a short reference audio (e.g. `speaker.wav`) to the Colab file panel, then run:

In [ ]:
request = TTSRequest(
    text="Hello from Auralis running on vLLM V1 Engine.",
    language="en",
    speaker_files=["speaker.wav"],
)

output = tts.generate_speech(request)
output.save("out.wav")

from IPython.display import Audio
Audio("out.wav")

## 4. Shutdown

Always shut the engine down before the runtime disconnects — this flushes hidden-state temp files and terminates vLLM's background workers cleanly.

In [ ]:
import asyncio
asyncio.get_event_loop().run_until_complete(tts.shutdown())